# Phase 5 & 6 — Feature Extraction and Train/Val/Test Split
## Heart Disease Detection using Phonocardiogram (PCG) and IoT
**BTech Major Project | Data Science | Galgotias College of Engineering & Technology**

**Authors:** Princee Singh (2300971630044) · Priyanshu Kumar (2300971630045)
**Supervisor:** Dr. Kalpna Sagar

---

### Phase 5 — Feature Extraction
Convert each 4-second window (8,000 raw float32 samples) into two
complementary 2D representations for model input:
- **Log-mel spectrogram:** captures time-frequency energy distribution
- **MFCCs:** compact spectral envelope features

### Phase 6 — Train / Validation / Test Split
Divide the 61,575 windows into three non-overlapping splits using a
**group-aware strategy** that prevents data leakage between patients.

### Why Both Phases Share One Notebook
Feature extraction produces the arrays that are immediately split —
keeping both phases together avoids re-loading 3.8 GB of features between
sessions and ensures the split indices are computed on the exact same
feature arrays used for training.


---
# PHASE 5 — Feature Extraction

## Why Log-Mel Spectrograms?

Raw waveforms contain 8,000 numbers per window — most of which encode
redundant information. A 2D time-frequency representation is more compact
and lets a CNN learn spatial patterns (shapes, textures) in the
frequency-time plane rather than fitting 1D sequential patterns.

**Why mel scale?** The mel scale is a perceptual frequency scale that
compresses high frequencies (where PCG has little content) and expands
low frequencies (where S1/S2 and murmurs live). This gives the model
finer resolution exactly where it matters.

**Why log (dB) scale?** Human auditory perception and cardiac acoustics
both follow roughly logarithmic amplitude relationships. Converting power
to dB scale makes the dynamic range more uniform across clips and
frequency bands — the model does not have to learn to ignore orders-of-
magnitude amplitude differences that carry no diagnostic information.

**fmin=20, fmax=400:** Critically, we constrain the mel filterbank to
exactly the 20–400 Hz bandpass range applied in Phase 3. Without this
constraint, the default mel filterbank covers 0–1000 Hz (half the Nyquist
at 2 kHz), wasting 60% of the 64 mel bands on frequencies that are
empty after bandpass filtering. Adding fmin/fmax maps all 64 bands to the
diagnostically relevant frequency range, giving the model much finer
frequency resolution where the heart sounds actually are.

This was discovered during visual inspection of the first spectrograms:
all energy was compressed into the bottom 20% of the plot (below ~100 Hz),
with the remaining 80% uniformly dark. After adding fmin/fmax, the energy
spreads across the full plot height.


## Step 1 — Setup: Mount Drive and Load Windows

**Session restart note:** This notebook begins a new Colab session.
The windows zip from Phase 4 must be unzipped to local disk again.
The `features.zip` from Phase 5's batch run is what subsequent phases
use — the windows are an intermediate that can be discarded after this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/pcg-project'

import numpy as np, pandas as pd, os, librosa
import matplotlib.pyplot as plt
from tqdm import tqdm

!mkdir -p /content/work/windows
!unzip -q "{PROJECT}/data/windows.zip" -d /content/work/windows_unzip

# Locate actual landing path (cd-then-zip in Phase 4 → clean relative path)
!find /content/work/windows_unzip -name "*.npy" | head -3
!echo "Total window files:"
!find /content/work/windows_unzip -name "*.npy" | wc -l


Mounted at /content/drive
/content/work/windows_unzip/windows/0.npy
/content/work/windows_unzip/windows/1.npy
/content/work/windows_unzip/windows/2.npy
Total window files:
61575


In [ ]:
# Load windows manifest and re-derive window paths for this session
windows_manifest = pd.read_csv(f"{PROJECT}/data/windows_manifest.csv")
WINDOWS_DIR = '/content/work/windows_unzip/windows'

windows_manifest['window_path'] = windows_manifest.index.map(
    lambda i: f"{WINDOWS_DIR}/{i}.npy"
)

missing = windows_manifest['window_path'].apply(lambda p: not os.path.exists(p)).sum()
print(f"Windows manifest shape: {windows_manifest.shape}")
print(f"Missing window files:   {missing} / {len(windows_manifest)}")
assert missing == 0, "Window files not found — check unzip path"


Windows manifest shape: (61575, 7)
Missing window files:   0 / 61575


---
## Step 2 — Feature Extraction Parameters and Functions


In [ ]:
TARGET_SR  = 2000   # all clips resampled to this in Phase 3
N_FFT      = 256    # FFT window = 128 ms at 2 kHz — roughly one heartbeat cycle
HOP_LENGTH = 32     # step = 16 ms — fine enough to resolve S1/S2 onset timing
N_MELS     = 64     # mel bands covering 20–400 Hz
N_MFCC     = 20     # first 20 MFCCs capture ~99% of spectral envelope variance


def extract_logmel(window, sr=TARGET_SR):
    """
    Compute log-mel spectrogram for a 4-second PCG window.

    Output shape: (64, 251) — 64 mel bands x 251 time frames
    (251 = floor((8000 - 256) / 32) + 1)

    Key parameter: fmin=20, fmax=400 constrains the mel filterbank to the
    diagnostically relevant frequency range (matching our Phase 3 bandpass
    filter). Without this, ~60% of the 64 mel bands map to 400–1000 Hz,
    which is empty after bandpass filtering — wasting model capacity on
    uninformative features.

    ref=np.max normalises each spectrogram to its own loudest bin.
    This is equivalent to the amplitude normalisation in Phase 3 but
    in the frequency domain — ensures no single recording's absolute
    volume dominates the representation.
    """
    mel = librosa.feature.melspectrogram(
        y=window, sr=sr, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS,
        fmin=20, fmax=400
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def extract_mfcc(window, sr=TARGET_SR):
    """
    Compute MFCCs (Mel-Frequency Cepstral Coefficients).

    Output shape: (20, 251) — 20 coefficients x 251 time frames

    MFCCs are a compact representation of the spectral envelope — the
    'shape' of the spectrum at each time frame. The first coefficient
    captures overall energy, subsequent ones capture increasingly fine
    spectral shape detail. 20 coefficients is standard in medical audio
    literature and captures the shape of murmurs and S1/S2 waveforms
    without overfitting to fine spectral detail.

    Both log-mel and MFCC features are extracted and saved — the model
    training (Phase 7) uses log-mel as the primary input, while MFCC
    is available as an alternative or ensemble feature.
    """
    return librosa.feature.mfcc(
        y=window, sr=sr, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH
    ).astype(np.float32)


---
## Step 3 — Visual Validation: Spectrogram Inspection

**This is the most important validation step in Phase 5.**
Before running the full 61,575-window batch, visually confirm that the
spectrograms show meaningful structure. Specifically we look for:

1. **Periodic vertical structure** — bright vertical bands repeating at
   the heart rate (~1 per second at 60 bpm) corresponding to S1/S2 events
2. **Dark gaps between beats** (for normal/absent recordings) —
   confirming the model will see a clear rhythmic pattern
3. **Full use of the y-axis frequency range** (17–403 Hz after fmin/fmax)
   — if energy is compressed to the bottom 20%, the fmin/fmax constraint
   is not working

**Iteration note:** The first spectrogram visualisation (without fmin/fmax)
showed all energy compressed into the bottom sliver of the plot, with the
upper 80% uniformly dark. Adding `fmin=20, fmax=400` to `extract_logmel`
redistributed all 64 mel bands across the diagnostically relevant range,
making the beat structure and murmur content visually clear.


In [ ]:
samples = windows_manifest.groupby('label_binary').sample(2, random_state=4)
fig, axes = plt.subplots(len(samples), 2, figsize=(14, 3 * len(samples)))

for i, (_, row) in enumerate(samples.iterrows()):
    w      = np.load(row['window_path'])
    logmel = extract_logmel(w)
    mfcc   = extract_mfcc(w)

    librosa.display.specshow(
        logmel, sr=TARGET_SR, hop_length=HOP_LENGTH,
        x_axis='time', y_axis='mel', fmin=20, fmax=400, ax=axes[i][0]
    )
    axes[i][0].set_title(f"Log-mel | {row['label_mapped']} | {row['source']}")

    librosa.display.specshow(
        mfcc, sr=TARGET_SR, hop_length=HOP_LENGTH,
        x_axis='time', ax=axes[i][1]
    )
    axes[i][1].set_title(f"MFCC | {row['label_mapped']} | {row['source']}")

plt.tight_layout()
plt.show()


### Visual Inspection Findings

**Row 1 — normal (physionet2016):**
Dense, consistent horizontal energy bands with periodic vertical
brightening at each heartbeat. The model will see this as a regular,
repeating pattern — easy to learn as "normal."

**Row 2 — absent (circor2022):**
Sharp dark vertical stripes (silence between beats) with bright bursts at
beat onsets. This recording has very clearly defined S1/S2 events —
approximately 6 beats visible in 4 seconds (~90 bpm). Classic normal PCG
spectrogram pattern.

**Row 3 — abnormal (physionet2016):**
Silent lead-in for ~0.8 seconds (stethoscope not yet fully placed), then
two large energy bursts. This is an unusual but valid recording — the model
must learn that the burst pattern, not the silence, carries the diagnostic
information.

**Row 4 — abnormal (physionet2016) — most informative:**
The inter-beat regions are NOT fully dark — there is diffuse, lower-level
energy spread between the sharp S1/S2 beat onsets. This smeared inter-beat
energy is the **acoustic signature of a murmur** — continuous turbulent
blood flow generating sound in the systolic or diastolic phase. This is
exactly the pattern the model needs to learn to distinguish from Row 2's
clean inter-beat silence.

**Key contrast for the report:**
- Normal/absent: dark inter-beat gaps, energy only at S1/S2 onsets
- Abnormal/present: energy present between beats (murmur content)
- This distinction is visible to the human eye in the spectrogram —
  confirming that the log-mel representation carries sufficient
  discriminative information for the classifier


---
## Step 4 — Full Feature Extraction Batch

Extract log-mel spectrograms and MFCCs for all 61,575 windows.
Both feature types are stacked into single NumPy arrays for efficient
storage and loading — much faster than reading 61,575 individual files
during training (a single `np.load` for the stacked array vs. 61,575
separate disk reads per epoch).

**Expected shapes:**
- `logmel_arr`: (61575, 64, 251) — ~3.8 GB as float32
- `mfcc_arr`:   (61575, 20, 251) — ~1.2 GB as float32
- `labels_arr`: (61575,) — binary labels aligned to array rows


In [ ]:
os.makedirs('/content/work/features', exist_ok=True)
logmel_features, mfcc_features, valid_indices = [], [], []
failed = 0

for i, row in tqdm(windows_manifest.iterrows(), total=len(windows_manifest)):
    try:
        w = np.load(row['window_path'])
        logmel_features.append(extract_logmel(w))
        mfcc_features.append(extract_mfcc(w))
        valid_indices.append(i)
    except Exception:
        failed += 1

logmel_arr = np.stack(logmel_features)
mfcc_arr   = np.stack(mfcc_features)
labels_arr = windows_manifest.loc[valid_indices, 'label_binary'].values

print(f"Log-mel shape : {logmel_arr.shape}")
print(f"MFCC shape    : {mfcc_arr.shape}")
print(f"Labels shape  : {labels_arr.shape}")
print(f"Failed        : {failed}")

np.save('/content/work/features/logmel.npy', logmel_arr)
np.save('/content/work/features/mfcc.npy',   mfcc_arr)
np.save('/content/work/features/labels.npy', labels_arr)


100%|██████████| 61575/61575 [08:20<00:00, 123.09it/s]
Log-mel shape : (61575, 64, 251)
MFCC shape    : (61575, 20, 251)
Labels shape  : (61575,)
Failed        : 0


### Result
- **61,575 / 61,575 windows processed — 0 failures** ✅
- **Log-mel array:** (61575, 64, 251) — each window is a 64×251 image
- **MFCC array:** (61575, 20, 251) — each window is a 20×251 matrix
- **Runtime:** ~8.3 minutes on Colab CPU

The 251 time frames come from:
`floor((8000 - 256) / 32) + 1 = floor(7744/32) + 1 = 242 + 1 = 243`
Wait — actually: `floor((8000 - N_FFT) / HOP_LENGTH) + 1`
= `floor(7744 / 32) + 1 = 242 + 1 = 243`... librosa centers frames and
pads by default, yielding 251 frames. This is consistent across all windows
(all are exactly 8,000 samples) so the shape is uniform — no padding needed
at the model input level.


---
## Step 5 — Save Manifest and Backup to Drive


In [ ]:
valid_manifest = windows_manifest.loc[valid_indices].reset_index(drop=True)
valid_manifest.to_csv(f"{PROJECT}/data/features_manifest.csv", index=False)
print(f"Features manifest saved: {valid_manifest.shape}")

# cd-then-zip for clean relative paths in the archive
!cd /content/work && zip -qr /content/features.zip features/
!cp /content/features.zip "{PROJECT}/data/"
print("Backup complete.")
!ls -lh "{PROJECT}/data/features.zip"


Features manifest saved: (61575, 7)
Backup complete.
-rw------- 1 root root 3.8G features.zip


### Storage Management Note
At this point the features.zip (3.8 GB) is the only file that must be
retained in Drive. The earlier intermediate zips can be deleted to free
space:
- `raw/physionet2016.zip` (1.0 GB) — re-downloadable from PhysioNet
- `raw/circor2022.zip` (451 MB) — re-downloadable from PhysioNet
- `processed/processed_audio.zip` (1.1 GB) — reproducible from Phase 3
- `windows.zip` (1.8 GB) — reproducible from Phase 4

In practice, Drive storage (15 GB free tier) was exhausted after saving
features.zip, requiring deletion of the intermediate zips. This is
expected and acceptable — the pipeline is fully reproducible from the
raw datasets using the code in Phases 1–5.


---
# PHASE 6 — Train / Validation / Test Split

## Design Principles

### Group-Aware Splitting
Standard random splitting assigns windows independently — this would
split windows from the same patient's recording across train and test.
Since our windows come from the same parent clip (and for CirCor, multiple
clips from the same patient), this creates **data leakage**: the model
could learn patient-specific characteristics (microphone placement, body
habitus, resting heart rate) rather than disease-relevant patterns.

We use `GroupShuffleSplit` from scikit-learn, which guarantees that all
windows sharing the same `group_id` (same patient for CirCor, same
recording for PhysioNet) land in exactly one split.

### Split Ratios
- **Train: 70%** (41,706 windows) — model parameter fitting
- **Val: 15%** (10,091 windows) — hyperparameter tuning, early stopping
- **Test: 15%** (9,778 windows) — final held-out evaluation (never seen during training)

The test set is held out completely until the model is fully trained and
all hyperparameters are fixed. Reporting metrics on the test set is the
honest measure of generalisation performance.


## Step 6 — Load Features for Splitting

In a new session, load the features_v2.zip (which includes the final
features plus split indices bundled together from Step 9 below).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/pcg-project'

import numpy as np, pandas as pd

!mkdir -p /content/work/features
!unzip -q "{PROJECT}/data/features.zip" -d /content/work/features_unzip

# Confirm landing path
!find /content/work/features_unzip -name "*.npy" | head -5


Mounted at /content/drive
/content/work/features_unzip/features/mfcc.npy
/content/work/features_unzip/features/labels.npy
/content/work/features_unzip/features/logmel.npy


In [ ]:
FEAT_DIR = '/content/work/features_unzip/features'

logmel = np.load(f'{FEAT_DIR}/logmel.npy')
mfcc   = np.load(f'{FEAT_DIR}/mfcc.npy')
labels = np.load(f'{FEAT_DIR}/labels.npy')

features_manifest = pd.read_csv(f"{PROJECT}/data/features_manifest.csv")

print(f"logmel  : {logmel.shape}")
print(f"mfcc    : {mfcc.shape}")
print(f"labels  : {labels.shape}")
print(f"manifest: {features_manifest.shape}")


logmel  : (61575, 64, 251)
mfcc    : (61575, 20, 251)
labels  : (61575,)
manifest: (61575, 7)


---
## Step 7 — Group-Aware Train / Val / Test Split


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups  = features_manifest['group_id'].values
sources = features_manifest['source'].values

# Step A: carve out 15% as held-out test set
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(gss_test.split(logmel, labels, groups=groups))

# Step B: from remaining 85%, carve out ~15% of total as validation
# 0.176 * 85% ≈ 15% of total
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=42)
train_idx, val_idx = next(
    gss_val.split(logmel[trainval_idx], labels[trainval_idx],
                  groups=groups[trainval_idx])
)
train_idx = trainval_idx[train_idx]
val_idx   = trainval_idx[val_idx]

print(f"Train : {len(train_idx):,} windows")
print(f"Val   : {len(val_idx):,} windows")
print(f"Test  : {len(test_idx):,} windows")
print(f"Total : {len(train_idx)+len(val_idx)+len(test_idx):,}")


Train : 41,706 windows
Val   : 10,091 windows
Test  : 9,778 windows
Total : 61,575


---
## Step 8 — Verify Zero Group Overlap (Leakage Check)

This is the critical correctness check for the split. Any overlap between
splits would mean the same patient's data appears in both training and
evaluation — a form of data leakage that silently inflates accuracy metrics.
All three overlap counts must be exactly 0.


In [ ]:
train_groups = set(groups[train_idx])
val_groups   = set(groups[val_idx])
test_groups  = set(groups[test_idx])

print(f"Train/Val overlap  : {len(train_groups & val_groups)} groups (must be 0)")
print(f"Train/Test overlap : {len(train_groups & test_groups)} groups (must be 0)")
print(f"Val/Test overlap   : {len(val_groups & test_groups)} groups (must be 0)")

assert len(train_groups & val_groups)   == 0, "LEAKAGE: train/val overlap!"
assert len(train_groups & test_groups)  == 0, "LEAKAGE: train/test overlap!"
assert len(val_groups   & test_groups)  == 0, "LEAKAGE: val/test overlap!"
print("\n✅ Zero group overlap confirmed — no data leakage")


Train/Val overlap  : 0 groups (must be 0)
Train/Test overlap : 0 groups (must be 0)
Val/Test overlap   : 0 groups (must be 0)

✅ Zero group overlap confirmed — no data leakage


### Finding — Leakage Check Passed ✅
Zero overlap across all three split pairs confirms that GroupShuffleSplit
correctly assigned every patient's/recording's windows to exactly one split.
This is a stronger guarantee than a standard random split would provide.


---
## Step 9 — Check Class Balance Per Split


In [ ]:
print("Class balance per split:")
print(f"{'Split':<8} {'N':>8} {'%Abnormal':>12}")
print("-" * 32)
for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    pos = labels[idx].mean()
    print(f"{name:<8} {len(idx):>8,} {pos:>11.2%}")
print(f"{'Overall':<8} {len(labels):>8,} {labels.mean():>11.2%}")


Class balance per split:
Split          N    %Abnormal
--------------------------------
Train     41,706       20.90%
Val       10,091       20.78%
Test       9,778       17.17%
Overall   61,575       20.29%


### Finding — Class Balance Analysis
| Split | Windows | % Abnormal |
|---|---|---|
| Train | 41,706 | 20.90% |
| Val | 10,091 | 20.78% |
| **Test** | **9,778** | **17.17%** |
| Overall | 61,575 | 20.29% |

**Train and Val are nearly identical (20.90% vs 20.78%)** — ideal for
training monitoring. The model's validation metrics will accurately reflect
training progress.

**Test is slightly lower (17.17% vs 20.29%)** — a small but expected
deviation from group-aware splitting. When groups vary in size and class
composition, exact stratification is impossible while maintaining zero
group overlap. The 3% difference is not large enough to meaningfully
bias the test evaluation.

**Implication:** The test set will have slightly more normal recordings
than training. This means test accuracy may be marginally higher than val
accuracy for a model that is conservative (predicts normal by default) —
but test AUC will be an accurate measure of ranking ability regardless
of this composition difference.


---
## Step 10 — Save Split Indices and Update Features Zip


In [ ]:
import os

# Save split indices as .npy (small files — a few hundred KB each)
np.save('/content/work/features/train_idx.npy', train_idx)
np.save('/content/work/features/val_idx.npy',   val_idx)
np.save('/content/work/features/test_idx.npy',  test_idx)

# Save per-split manifests to Drive (for inspection/debugging)
for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    split_df = features_manifest.iloc[idx].copy()
    split_df.to_csv(f"{PROJECT}/data/split_{name}.csv", index=False)

print("Split CSVs saved to Drive")
print("Split index .npy files saved to local features folder")


Split CSVs saved to Drive
Split index .npy files saved to local features folder


In [ ]:
# Bundle split indices into features zip for one-step restore in future sessions
!cp /content/work/features/train_idx.npy \
    /content/work/features/val_idx.npy \
    /content/work/features/test_idx.npy \
    /content/work/features_unzip/features/

!cd /content/work/features_unzip && zip -qr /content/features_v2.zip features/
!cp /content/features_v2.zip "{PROJECT}/data/"
!ls -lh "{PROJECT}/data/features_v2.zip"


-rw------- 1 root root 3.8G Jul  9 09:00 /content/drive/MyDrive/pcg-project/data/features_v2.zip


---
## Phases 5 & 6 Summary

### Phase 5 — Feature Extraction
| Step | Action | Result |
|---|---|---|
| 1 | Load windows from Drive | 61,575 window files confirmed ✅ |
| 2 | Define feature functions | Log-mel (fmin=20, fmax=400) + MFCC |
| 3 | Visual validation | Murmur inter-beat energy visible in spectrograms ✅ |
| 4 | Full batch extraction | 61,575/61,575 — 0 failures · ~8.3 min |
| 5 | Save arrays + backup | logmel (61575,64,251) · mfcc (61575,20,251) |

### Phase 6 — Train/Val/Test Split
| Step | Action | Result |
|---|---|---|
| 6 | Load feature arrays | Shapes confirmed (61575, 64, 251) ✅ |
| 7 | Group-aware split | 41,706 train / 10,091 val / 9,778 test |
| 8 | Leakage check | **Zero overlap** across all 3 split pairs ✅ |
| 9 | Class balance check | Train/Val ~20.9% · Test 17.2% abnormal |
| 10 | Save indices + rezip | features_v2.zip (3.8 GB) backed up ✅ |

### Key Design Decisions

**Feature Extraction:**
- `fmin=20, fmax=400` in mel filterbank — all 64 bands map to the
  physiologically relevant PCG frequency range
- Log (dB) scale — compresses dynamic range for more uniform model input
- MFCC extracted alongside log-mel — available as ensemble/comparison feature

**Split Strategy:**
- `GroupShuffleSplit` — guarantees patient-level separation
- Two-stage split (test first, then val from remainder) — ensures test
  is truly independent of all training decisions
- Zero leakage confirmed by explicit group intersection check with assert

### Next Step → Phase 7: Model Training
The feature arrays and split indices from `features_v2.zip` are the only
inputs needed for model training. The architecture — a CNN-BiLSTM with
attention — will treat each log-mel spectrogram as a 2D image, extract
spatial features with convolution, model temporal dependencies with a
bidirectional LSTM, and classify via an attention-weighted pooling head.
